# main_program.ipynb — Web Interface

Run all cells in this notebook to start the web server at **http://localhost:8848**.

This notebook:
1. Imports all pipeline functions from `demo.ipynb`.
2. Defines `web_work()` — the PyWebIO application that handles the full user interaction.
3. Starts the PyWebIO server on the configured port.

**Do not edit pipeline logic here** — all analysis code lives in `demo.ipynb`.

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import import_ipynb          # allows importing demo.ipynb as a Python module
from demo import *           # load all pipeline functions from demo.ipynb

# PyWebIO — web interface components
from pywebio import start_server
from pywebio.input import (
    input as web_input,
    input_group,
    select,
    TEXT,
)
from pywebio.output import (
    put_html,
    put_text,
    put_markdown,
    put_processbar,
    set_processbar,
    put_file,
    put_image,
    put_success,
    put_error,
    put_warning,
    put_info,
    put_collapse,
    use_scope,
    remove,
    clear,
    toast,
    put_loading,
)
from pywebio.session import set_env

import os
from datetime import datetime

In [ ]:
# ---------------------------------------------------------------------------
# Shared CSS injected into every page load
# ---------------------------------------------------------------------------
PAGE_STYLE = """
<style>
  /* Overall page */
  body { font-family: 'Segoe UI', Arial, sans-serif; background: #f4f6fb; }

  /* Hero banner */
  .hero {
    background: linear-gradient(135deg, #1a237e 0%, #283593 60%, #3949ab 100%);
    color: #fff;
    padding: 36px 32px 28px;
    border-radius: 12px;
    margin-bottom: 24px;
    box-shadow: 0 4px 18px rgba(26,35,126,0.18);
  }
  .hero h1 { margin: 0 0 8px; font-size: 2rem; letter-spacing: 0.5px; }
  .hero p  { margin: 0; font-size: 1.05rem; opacity: 0.88; line-height: 1.6; }
  .hero .badge {
    display: inline-block;
    background: rgba(255,255,255,0.18);
    border: 1px solid rgba(255,255,255,0.35);
    border-radius: 20px;
    padding: 3px 12px;
    font-size: 0.82rem;
    margin-right: 6px;
    margin-top: 12px;
  }

  /* Workflow steps */
  .workflow {
    display: flex;
    flex-wrap: wrap;
    gap: 10px;
    margin-bottom: 20px;
  }
  .step {
    background: #fff;
    border-left: 4px solid #3949ab;
    border-radius: 6px;
    padding: 10px 14px;
    flex: 1 1 180px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.07);
    font-size: 0.9rem;
  }
  .step .num {
    font-weight: 700;
    color: #3949ab;
    font-size: 1.1rem;
  }

  /* Demo card */
  .demo-card {
    background: #fff;
    border: 1px solid #e0e0e0;
    border-radius: 8px;
    padding: 16px 20px;
    margin-bottom: 12px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.06);
  }
  .demo-card h4 { margin: 0 0 8px; color: #1a237e; }
  .demo-card code {
    display: block;
    background: #f5f5f5;
    border-radius: 4px;
    padding: 8px;
    font-size: 0.82rem;
    word-break: break-all;
    margin-top: 6px;
  }
  .demo-label {
    font-size: 0.78rem;
    font-weight: 600;
    color: #555;
    text-transform: uppercase;
    letter-spacing: 0.5px;
    margin-top: 10px;
  }

  /* Result section */
  .result-box {
    background: #fff;
    border: 1px solid #c8e6c9;
    border-radius: 8px;
    padding: 20px 24px;
    margin-top: 16px;
    box-shadow: 0 1px 6px rgba(0,0,0,0.07);
  }
  .result-box h3 { margin: 0 0 12px; color: #2e7d32; }

  /* Footer */
  .footer {
    text-align: center;
    color: #888;
    font-size: 0.85rem;
    padding: 18px 0 8px;
    border-top: 1px solid #e0e0e0;
    margin-top: 28px;
  }
</style>
"""

In [ ]:
def web_work():
    """
    Main PyWebIO application function.

    Lifecycle
    ---------
    1. Render the landing page: hero banner, pipeline overview, and demo examples.
    2. Collect and validate user inputs via an input form.
    3. Run the pipeline with a live progress bar.
    4. Display result images and offer a download link for the full archive.

    This function is passed to PyWebIO's start_server() and is called once
    per browser session.
    """
    # Set the browser tab title
    set_env(title='Novel Cis-Element Discovery')

    # ── Inject shared CSS ────────────────────────────────────────────────────
    put_html(PAGE_STYLE)

    # ── Hero banner ──────────────────────────────────────────────────────────
    with use_scope('landing'):
        put_html("""
        <div class="hero">
          <h1>🔬 Novel Cis-Element Discovery Pipeline</h1>
          <p>
            Identify conserved transcriptional regulatory elements across a bacterial taxon.<br>
            Provide a set of co-regulated protein IDs — the pipeline finds their shared
            upstream DNA motifs through comparative genomics and <em>de novo</em> motif discovery.
          </p>
          <span class="badge">DIAMOND BLASTP</span>
          <span class="badge">MEME Suite</span>
          <span class="badge">CD-HIT</span>
          <span class="badge">NCBI RefSeq</span>
        </div>
        """)

        # ── Pipeline overview ─────────────────────────────────────────────────
        put_html("""
        <div class="workflow">
          <div class="step"><span class="num">① </span>Download reference genomes (NCBI)</div>
          <div class="step"><span class="num">② </span>Extract proteins &amp; promoters</div>
          <div class="step"><span class="num">③ </span>DIAMOND homolog search</div>
          <div class="step"><span class="num">④ </span>Cluster promoters (CD-HIT)</div>
          <div class="step"><span class="num">⑤ </span>Motif discovery (MEME)</div>
          <div class="step"><span class="num">⑥ </span>Filter &amp; visualise (R)</div>
        </div>
        """)

        # ── Demo examples ─────────────────────────────────────────────────────
        put_markdown('### 📋 Demo Examples')
        put_html("""
        <div class="demo-card">
          <h4>Example 1 — <em>Acinetobacter</em> DdaA (DNA Damage Response)</h4>
          <div class="demo-label">Taxonomy</div>
          <code>Acinetobacter</code>
          <div class="demo-label">NCBI Protein IDs (copy all into the form below)</div>
          <code>WP_005017521.1,WP_005405730.1,WP_034675625.1,WP_111280119.1,WP_111280136.1,
WP_005017085.1,WP_101160495.1,WP_005405985.1,WP_111280201.1,WP_111280215.1,
WP_111280241.1,WP_111280243.1,WP_111280245.1,WP_111280252.1,WP_111280336.1,
WP_111280342.1,WP_005016405.1,WP_005016170.1,WP_111280466.1,WP_111280719.1,
WP_111280773.1,WP_005014868.1,WP_111280906.1,WP_111280918.1,WP_005014692.1,
WP_005014686.1,WP_005405657.1,WP_005025682.1,WP_005404176.1,WP_005014561.1,
WP_111281284.1,WP_005014193.1,WP_005020133.1,WP_111281445.1,WP_005019893.1,
WP_111281515.1,WP_111281570.1,WP_111281616.1,WP_005026719.1,WP_111281678.1,
WP_005019389.1,WP_111281740.1,WP_005019323.1,WP_005027049.1,WP_005027120.1,
WP_111281831.1,WP_005406549.1,WP_111281978.1,WP_111282072.1,WP_111282809.1,
WP_111282131.1,WP_111282142.1,WP_005018919.1,WP_111282175.1,WP_111282201.1,
WP_005018794.1,WP_046205772.1,WP_005027688.1,WP_106439128.1,WP_005018365.1,
WP_111282411.1,WP_111282415.1,WP_005024277.1,WP_005017910.1,WP_111282564.1,
WP_111282582.1,WP_005024026.1,WP_111282679.1,WP_026056665.1</code>
          <div class="demo-label">Genes per combination</div>
          <code>1</code>
        </div>

        <div class="demo-card">
          <h4>Example 2 — <em>Deinococcus</em> DdrO (DNA Damage Response)</h4>
          <div class="demo-label">Taxonomy</div>
          <code>Deinococcus</code>
          <div class="demo-label">NCBI Protein IDs</div>
          <code>WP_012695046.1,WP_012694483.1,WP_049760561.1,WP_012692255.1,WP_012693173.1,
WP_162485426.1,WP_012694690.1,WP_012693159.1,WP_012692693.1,WP_012691930.1,
WP_012693877.1,WP_012693497.1</code>
          <div class="demo-label">Genes per combination</div>
          <code>1</code>
        </div>
        """)

        # ── Separator before the form ─────────────────────────────────────────
        put_html('<hr style="border:none;border-top:1px solid #e0e0e0;margin:20px 0">')
        put_markdown('### 🚀 Run the Pipeline')

    # ── Input form ───────────────────────────────────────────────────────────
    info = input_group(
        'Novel Cis-Element Discovery — Input Form',
        [
            web_input(
                'Bacterial Taxonomy',
                name='tax',
                type=TEXT,
                placeholder='e.g. Acinetobacter',
                help_text=(
                    'NCBI taxon name for the bacterial group. '
                    'Must have 5–100 reference genomes in RefSeq.'
                ),
                required=True,
            ),
            web_input(
                'NCBI Protein IDs (comma-separated)',
                name='protein_id',
                type=TEXT,
                placeholder='WP_005017521.1, WP_005405730.1, ...',
                help_text=(
                    'RefSeq protein accessions of co-regulated genes '
                    '(max 80 entries). Spaces around commas are allowed.'
                ),
                required=True,
            ),
            web_input(
                'Genes per combination',
                name='set_length',
                type=TEXT,
                placeholder='1',
                help_text=(
                    '0 = comparative genomics only (all promoters merged, one MEME run)  |  '
                    '1 = gene-specific motif discovery  |  '
                    '2 = pairwise combinations  |  '
                    '3 = three-way combinations'
                ),
                required=True,
            ),
        ]
    )

    # ── Clear the landing page once the user submits ──────────────────────────
    remove('landing')
    put_html(PAGE_STYLE)   # re-inject CSS after clear

    # ── Parse and echo inputs ─────────────────────────────────────────────────
    ntime        = str(datetime.now()).replace(' ', '_')          # unique job ID
    taxonomy     = info['tax'].strip().replace(' ', '')
    proteins_raw = info['protein_id'].replace(' ', '')
    protein_list = [p.strip() for p in proteins_raw.split(',') if p.strip()]
    set_length   = int(info['set_length'].strip())

    put_html("""
    <div style="background:#fff;border:1px solid #e0e0e0;border-radius:8px;
                padding:16px 20px;margin-bottom:16px;box-shadow:0 1px 4px rgba(0,0,0,0.06)">
      <b>📥 Job submitted</b><br>
      <span style="color:#555">Taxonomy:</span> <b>{tax}</b><br>
      <span style="color:#555">Proteins:</span> {n_prot} accessions<br>
      <span style="color:#555">Genes per combination:</span> <b>{sl}</b><br>
      <span style="color:#888;font-size:0.85rem">Job ID: {ntime}</span>
    </div>
    """.format(tax=taxonomy, n_prot=len(protein_list), sl=set_length, ntime=ntime))

    # ── Input validation ─────────────────────────────────────────────────────
    put_info('🔍 Validating inputs...')
    check_result = check_before_task(taxonomy, protein_list, set_length)

    if check_result != 'True':
        put_error('❌ Validation failed: ' + check_result)
        put_markdown(
            '**Please go back, correct the inputs, and resubmit.**  \n'
            'If you believe this is an error, contact shuang_s@zju.edu.cn.'
        )
        return   # stop the session gracefully

    put_success('✅ Validation passed — starting the pipeline.')

    # ── Progress bar ─────────────────────────────────────────────────────────
    put_processbar('bar')

    def progress(value: float, message: str):
        """Update the progress bar and print a status message."""
        set_processbar('bar', value)
        put_text('  ⏳ ' + message)

    # ── Pipeline execution ───────────────────────────────────────────────────
    gbk_dir = os.path.join(BASE_DIR, '0_gbk')

    # Steps 1–3 are skipped if the taxon database already exists (cached)
    if taxonomy not in os.listdir(gbk_dir):
        progress(0.1, 'Downloading reference genomes from NCBI...')
        download_gbk(taxonomy)

        progress(0.25, 'Extracting protein and promoter sequences...')
        extract_protein(taxonomy)

        progress(0.35, 'Building DIAMOND protein database...')
        diamond_task(taxonomy)
    else:
        progress(0.35, 'Using cached genome database for ' + taxonomy)

    progress(0.40, 'Searching for homologs and extracting promoters...')
    get_selected_protein_diamond(protein_list, taxonomy, ntime)

    progress(0.60, 'Running MEME motif discovery (this may take 10–30 min)...')
    combine_meme_work(ntime, set_length)

    progress(0.90, 'Filtering and visualising motifs with R...')
    motif_for_R(ntime)

    set_processbar('bar', 1.0)
    put_success('🎉 Pipeline complete!')

    # ── Package results ───────────────────────────────────────────────────────
    result_dir = os.path.join(BASE_DIR, '99_User_project', ntime, '3_meme_result')
    promoter_dir = os.path.join(BASE_DIR, '99_User_project', ntime, '2_promoter')
    os.chdir(result_dir)

    # Move promoter directory into results for archiving
    os.system('mv %s ./promoter' % promoter_dir)

    archive_name = '%s.tar.gz' % taxonomy
    os.system('tar -czvf %s *' % archive_name)

    # ── Result display ────────────────────────────────────────────────────────
    put_html('<div class="result-box"><h3>📊 Results</h3>')

    # Standard motif logo
    standard_img = os.path.join(result_dir, 'meme', 'novel_motif.png')
    if os.path.exists(standard_img):
        put_markdown('#### Standard Motifs')
        put_image(open(standard_img, 'rb').read())
    else:
        put_warning('ℹ️ No significant standard motifs were found at E-value < 1e-35.')

    # Palindrome motif logo
    pal_img = os.path.join(result_dir, 'pal', 'novel_motif.png')
    if os.path.exists(pal_img):
        put_markdown('#### Palindromic Motifs')
        put_image(open(pal_img, 'rb').read())
    else:
        put_warning('ℹ️ No significant palindromic motifs were found at E-value < 1e-20.')

    # Download link for the full results archive
    put_markdown('#### 📦 Download Full Results')
    archive_bytes = open(os.path.join(result_dir, archive_name), 'rb').read()
    put_file(archive_name, archive_bytes, 'Download %s' % archive_name)

    put_html('</div>')   # close .result-box

    # ── Footer ────────────────────────────────────────────────────────────────
    put_html("""
    <div class="footer">
      Novel Cis-Element Discovery Pipeline &mdash;
      Maintained by <a href="mailto:shuang_s@zju.edu.cn">shuang_s@zju.edu.cn</a> &mdash;
      <a href="https://github.com/songshuang1996/Novel-Sample" target="_blank">GitHub</a>
    </div>
    """)

In [ ]:
# ---------------------------------------------------------------------------
# Start the web server
# ---------------------------------------------------------------------------
# Load the port from config (default 8848 if config import fails)
try:
    import config
    _port = getattr(config, 'PORT', 8848)
except ImportError:
    _port = 8848

if __name__ == '__main__':
    print('Starting server at http://localhost:%d' % _port)
    # debug=True reloads the app on code changes during development
    start_server(web_work, port=_port, debug=False)